In [2]:
import os
import pandas as pd
import plotly.express as px
import numpy as np

os.chdir('C:/Users/user/OneDrive/Desktop/MyProjects/Project2')
df = pd.read_csv('data/master_clean.csv')
COLORS = {'SOFT':'#FF1801','MEDIUM':'#FFD600','HARD':'#EEEEEE'}
print(df.shape)

(4320, 9)


In [3]:
fig = px.scatter(df, x='TyreLife', y='LapTimeSeconds',
    color='Compound', color_discrete_map=COLORS,
    trendline='lowess', trendline_options={'frac':0.3},
    title='Tire Degradation — Lap Time vs. Tire Age Across All Races',
    labels={'TyreLife':'Tire Age (laps)','LapTimeSeconds':'Lap Time (s)'},
    opacity=0.3, facet_col='Compound')

fig.update_layout(plot_bgcolor='#0d0d0d',paper_bgcolor='#0d0d0d',font_color='#f0f0f0')

fig.show()

In [ ]:
df['TyreBin'] = (df['TyreLife'] // 5) * 5
deg_table = df.groupby(['Compound','TyreBin'])['LapTimeSeconds'].agg(
    AvgLapTime='mean', LapCount='count').reset_index()
deg_table = deg_table[deg_table['LapCount'] >= 5]

print(deg_table.head(10))
#binning each tire life into 5 lap intervals, then calculating the average lap time and count of laps in each bin for each compound. Finally, filtering out bins with fewer than 5 laps to ensure statistical significance.

  Compound  TyreBin  AvgLapTime  LapCount
0     HARD      0.0   93.564360       347
1     HARD      5.0   93.296817       623
2     HARD     10.0   93.509891       626
3     HARD     15.0   93.145435       542
4     HARD     20.0   93.340485       344
5     HARD     25.0   94.645865       208
6     HARD     30.0   95.954688       125
7     HARD     35.0   93.464014        74
8     HARD     40.0   93.197780        50
9   MEDIUM      0.0   96.549569       239


In [ ]:
fig2 = px.line(deg_table, x='TyreBin', y='AvgLapTime',
    color='Compound', color_discrete_map=COLORS, markers=True,
    title='Average Degradation per 5-Lap Bin',
    labels={'TyreBin':'Tire Age','AvgLapTime':'Avg Lap Time (s)'})
fig2.update_layout(plot_bgcolor='#0d0d0d',paper_bgcolor='#0d0d0d',font_color='#f0f0f0')
fig2.write_html('charts/degradation_curves.html')
fig2.show()
#This code creates a line chart showing the average lap time for each 5-lap bin of tire age, separated by compound. The chart is styled with a dark background and saved as an HTML file for easy sharing.

In [8]:
for compound in ['SOFT','MEDIUM','HARD']:
    c = deg_table[deg_table['Compound']==compound]
    if len(c) >= 2:
        rate = (c['AvgLapTime'].iloc[-1] - c['AvgLapTime'].iloc[0]) / (c['TyreBin'].iloc[-1] - c['TyreBin'].iloc[0])
        print(f"{compound}: ~{rate:.3f}s per lap ({rate*1000:.0f}ms/lap)")

SOFT: ~-0.089s per lap (-89ms/lap)
MEDIUM: ~-0.075s per lap (-75ms/lap)
HARD: ~-0.009s per lap (-9ms/lap)


The SOFT compound shows lap times decreasing by ~89ms per bin, but this reflects the 
overall race trend (fuel burning off) more than true tyre degradation, since SOFT tyres 
are rarely used beyond 20 laps. The HARD compound shows the clearest true degradation 
signal — staying flat for ~25 laps before degrading sharply around lap 30, suggesting 
the ideal HARD stint length is under 25-30 laps before performance drops.